In [1]:
from workspace.sources.local_datasets.preprocessing.tokenization import NLTKTokenizer
from workspace.sources.local_datasets.preprocessing.cleaning import NoiseReduction, Lemmatization, Stemming
from workspace.sources.local_datasets.preprocessing.encoders.encoders import BertBaseUncasedEncoder
from workspace.sources.experiments.metrics import Loss
from workspace.sources.experiments.experiment import Experiment
from workspace.sources.local_datasets.dataset import Dataset
from workspace.sources.local_datasets.preprocessing.pipelines import PreprocessingPipeline
from workspace.sources.models.transformers.bert_based_models import BERT
from workspace.sources.local_datasets.preprocessing.utils import truncate
from sklearn.model_selection import ParameterGrid
import mlflow

mlflow.set_tracking_uri('../../mlruns')

In [2]:
experiment_name = 'prefinal-bert-base-uncased-v2'

In [4]:
def conduct_model_experiments(dataset_name,
                              dataset_path,
                              dataset_params_grid,
                              model_hparams_grid):
    for dataset_params in dataset_params_grid:
        for model_hparams in model_hparams_grid:
            dataset = Dataset(dataset_name, dataset_path, **dataset_params)

            model = BERT(train_best_model_metric=Loss,
                         training_arguments=model_hparams)

            recovery_experiment = Experiment(
                name=experiment_name,
                dataset=dataset,
                model=model)
            recovery_experiment.run()

In [7]:
pipelines = [PreprocessingPipeline(name='minimal',
                                   iterable=[BertBaseUncasedEncoder()]),
             PreprocessingPipeline(name='noise_reduction',
                                   iterable=[NoiseReduction(),
                                             BertBaseUncasedEncoder()]),
             PreprocessingPipeline(name='noise_reduction_with_lemmatization',
                                   iterable=[NoiseReduction(), NLTKTokenizer(), Lemmatization(),
                                             BertBaseUncasedEncoder()]),
             PreprocessingPipeline(name='noise_reduction_with_stemming',
                                   iterable=[NoiseReduction(), NLTKTokenizer(), Stemming(),
                                             BertBaseUncasedEncoder()])
             ]
# optional - for testing purposes, if you want to run fast test on very small datasets
truncate(pipelines, fraction=0.05)

dataset_hparams_grid = ParameterGrid({'preprocessings_pipeline': pipelines})

print('Dataset Hyperparameters Grid Size: ', len(dataset_hparams_grid))

model_hparams_grid = ParameterGrid({'epochs': [10],
                                    'batch_size': [16],
                                    'learning_rate': [1e-05, 5e-05],
                                    'lr_scheduler': ['linear'],
                                    'optimizer': ['adamw_torch'],
                                    'weight_decay': [0.0, 1e-3],
                                    'early_stopping_patience': [3],
                                    'early_stopping_threshold': [0.01]})

print('Model Hyperparameters Grid Size: ', len(model_hparams_grid))
print('Total Hyperparameter Combinations: ', len(dataset_params_grid) * len(model_hparams_grid))

Dataset Hyperparameters Grid Size:  4
Model Hyperparameters Grid Size:  4
Total Hyperparameter Combinations:  16


### ReCOVery Dataset

In [ ]:
dataset_name = 'ReCOVery'
dataset_path = '../../sources/local_datasets/data/ReCOVery/recovery.csv'

conduct_model_experiments(dataset_name,
                          dataset_path,
                          dataset_params_grid,
                          dataset_params_grid)

### EUvsDisinfo Dataset

In [ ]:
dataset_name = 'EUvsDisinfo'
dataset_path = '../../sources/local_datasets/data/EUvsDisinfo/euvsdisinfo.csv'

conduct_model_experiments(dataset_name,
                          dataset_path,
                          dataset_params_dict,
                          model_hparams_dict)

### ISOT Dataset

In [13]:
dataset_name = 'ISOT'
dataset_path = '../../sources/local_datasets/data/ISOT/isot.csv'

conduct_model_experiments(dataset_name,
                          dataset_path,
                          dataset_params_dict,
                          model_hparams_dict)

2025-05-14 17:15:51,348 - Experiment - INFO - MLflow experiment initialized with ID: 847989095527904331
2025-05-14 17:15:51,349 - Experiment - INFO - Model signature: model(mn=bert-base-uncased,mmn=loss,e=10,bs=8,ebs=8,lr=0.0001,wd=0.01,esp=3,est=0.01)
2025-05-14 17:15:51,350 - Experiment - INFO - Dataset signature: dataset(dn=isot,tp=0.7,vp=0.15);pipeline([truncation(f=0.1,tt=top),bert_base_uncased_encoder(t=bert-base-uncased,t=true,p=max_length,tml=512,isiw=false)])
2025-05-14 17:15:51,351 - Experiment - INFO - Run signature: model(mn=bert-base-uncased,mmn=loss,e=10,bs=8,ebs=8,lr=0.0001,wd=0.01,esp=3,est=0.01);dataset(dn=isot,tp=0.7,vp=0.15);pipeline([truncation(f=0.1,tt=top),bert_base_uncased_encoder(t=bert-base-uncased,t=true,p=max_length,tml=512,isiw=false)])
2025-05-14 17:15:52,209 - Experiment - INFO - Run ID: 373bf99bb79c41adacac8934cb0dcea5
2025-05-14 17:15:52,209 - Experiment - INFO - Run name: model(mn=bert-base-uncased,mmn=loss,e=10,bs=8,ebs=8,lr=0.0001,wd=0.01,esp=3,est=0.

Map:   0%|          | 0/700 [00:00<?, ? examples/s]

Map:   0%|          | 0/700 [00:00<?, ? examples/s]

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

2025-05-14 17:15:58,246 - Experiment - INFO - Preprocessing split: val_set
2025-05-14 17:15:58,247 - Experiment - INFO - Preprocessed data does not exist, computing from scratch.


...

2025-05-14 18:18:22,148 - Experiment - INFO - Test metrics: {'test_loss': 0.266695499420166, 'test_accuracy': 0.9666666666666667, 'test_precision': 0.9857142857142858, 'test_recall': 0.9452054794520548, 'test_f1_score': 0.965034965034965, 'test_roc_auc': 0.9940402063689735, 'test_false_positives_rate': 0.012987012987012988, 'test_false_negatives_rate': 0.0547945205479452, 'test_runtime': 4.867, 'test_samples_per_second': 30.82, 'test_steps_per_second': 3.904, 'test_epoch': 4.0}
2025-05-14 18:18:22,169 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-14 18:18:22,169 - Experiment - INFO - Saving evaluation data in json...
2025-05-14 18:18:22,588 - Experiment - INFO - Successfully saved evaluation data.
2025-05-14 18:18:22,590 - Experiment - INFO - Evaluating model using f1_score metric...
2025-05-14 18:18:22,592 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-264
2025-05-14 18:18:23,449 - Experiment - INFO - Best entry according to validation metr

2025-05-14 18:18:28,310 - Experiment - INFO - Test metrics: {'test_loss': 0.1827411651611328, 'test_accuracy': 0.9666666666666667, 'test_precision': 0.9594594594594594, 'test_recall': 0.9726027397260274, 'test_f1_score': 0.9659863945578231, 'test_roc_auc': 0.9925280199252802, 'test_false_positives_rate': 0.03896103896103896, 'test_false_negatives_rate': 0.0273972602739726, 'test_runtime': 4.8614, 'test_samples_per_second': 30.855, 'test_steps_per_second': 3.908, 'test_epoch': 3.0}
2025-05-14 18:18:28,326 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-14 18:18:28,342 - Experiment - INFO - Saving evaluation data in json...
2025-05-14 18:18:28,678 - Experiment - INFO - Successfully saved evaluation data.
2025-05-14 18:18:28,678 - Experiment - INFO - Evaluating model using false_positives_rate metric...
2025-05-14 18:18:28,678 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-352
2025-05-14 18:18:29,014 - Experiment - INFO - Best entry according to 

2025-05-14 18:18:33,901 - Experiment - INFO - Test metrics: {'test_loss': 0.266695499420166, 'test_accuracy': 0.9666666666666667, 'test_precision': 0.9857142857142858, 'test_recall': 0.9452054794520548, 'test_f1_score': 0.965034965034965, 'test_roc_auc': 0.9940402063689735, 'test_false_positives_rate': 0.012987012987012988, 'test_false_negatives_rate': 0.0547945205479452, 'test_runtime': 4.8868, 'test_samples_per_second': 30.695, 'test_steps_per_second': 3.888, 'test_epoch': 4.0}
2025-05-14 18:18:33,933 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-14 18:18:33,933 - Experiment - INFO - Saving evaluation data in json...
2025-05-14 18:18:34,301 - Experiment - INFO - Successfully saved evaluation data.
2025-05-14 18:18:34,301 - Experiment - INFO - Evaluating model using roc_auc metric...
2025-05-14 18:18:34,301 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-264
2025-05-14 18:18:34,637 - Experiment - INFO - Best entry according to validation met

2025-05-14 18:18:39,524 - Experiment - INFO - Test metrics: {'test_loss': 0.1827411651611328, 'test_accuracy': 0.9666666666666667, 'test_precision': 0.9594594594594594, 'test_recall': 0.9726027397260274, 'test_f1_score': 0.9659863945578231, 'test_roc_auc': 0.9925280199252802, 'test_false_positives_rate': 0.03896103896103896, 'test_false_negatives_rate': 0.0273972602739726, 'test_runtime': 4.8874, 'test_samples_per_second': 30.691, 'test_steps_per_second': 3.888, 'test_epoch': 3.0}
2025-05-14 18:18:39,557 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-14 18:18:39,557 - Experiment - INFO - Saving evaluation data in json...
2025-05-14 18:18:39,955 - Experiment - INFO - Successfully saved evaluation data.
2025-05-14 18:18:39,958 - Experiment - INFO - Evaluating model using loss metric...
2025-05-14 18:18:39,960 - Experiment - INFO - Found checkpoint with best metrics at: checkpoint-264
2025-05-14 18:18:40,352 - Experiment - INFO - Best entry according to validation metri

2025-05-14 18:18:45,241 - Experiment - INFO - Test metrics: {'test_loss': 0.1827411651611328, 'test_accuracy': 0.9666666666666667, 'test_precision': 0.9594594594594594, 'test_recall': 0.9726027397260274, 'test_f1_score': 0.9659863945578231, 'test_roc_auc': 0.9925280199252802, 'test_false_positives_rate': 0.03896103896103896, 'test_false_negatives_rate': 0.0273972602739726, 'test_runtime': 4.8832, 'test_samples_per_second': 30.717, 'test_steps_per_second': 3.891, 'test_epoch': 3.0}
2025-05-14 18:18:45,276 - Experiment - INFO - Saving evaluation data in pickle...
2025-05-14 18:18:45,276 - Experiment - INFO - Saving evaluation data in json...
2025-05-14 18:18:45,655 - Experiment - INFO - Successfully saved evaluation data.
2025-05-14 18:18:45,655 - Experiment - INFO - Finished model evaluations stage.


### CZ Dataset

In [ ]:
dataset_name = 'cz_dataset'
dataset_path = '../../sources/local_datasets/data/cz_dataset/cz_dataset.csv'

conduct_model_experiments(dataset_name,
                          dataset_path,
                          dataset_params_dict,
                          model_hparams_dict)